# 04: Custom Knowledge Base with Vertex AI Search

This notebook demonstrates how to query a **custom knowledge base** stored in Vertex AI Search.

Unlike the Google Search tool (which searches the public web), `vertex_search` queries a
**private data store** that you control. The model's answer is grounded directly in the
retrieved document content — not the public internet and not the model's training data.

## What You'll Learn

1. How to call `vertex_search` directly and read the result
2. How grounding works — what the model can and cannot answer
3. How to wrap the tool for use inside an ADK agent

## Prerequisites

- `VERTEX_AI_DATASTORE_ID` set in your `.env` file
- `GOOGLE_CLOUD_LOCATION` set (default: `us-central1`)
- Authentication via Application Default Credentials (ADC) — automatic on GCE/Coder workspaces

In [1]:
import os
from pathlib import Path

from aieng.agent_evals.configs import Configs
from aieng.agent_evals.tools import create_vertex_search_tool, vertex_search
from dotenv import load_dotenv
from rich.console import Console
from rich.panel import Panel
from rich.table import Table


if Path("").absolute().name == "eval-agents":
    print(f"Working directory: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"Working directory set to: {Path('').absolute()}")

load_dotenv(verbose=True)
console = Console(width=100)

Working directory set to: /Users/amritkrishnan/src/eval-agents


## 1. Configuration

The tool reads its configuration from the environment. Let's check what's loaded.

In [2]:
config = Configs()  # type: ignore[call-arg]

if not config.vertex_datastore_id:
    console.print("[red]✗[/red] VERTEX_AI_DATASTORE_ID is not set.")
    console.print("[dim]Set it in your .env file and re-run this cell.[/dim]")
else:
    cfg_table = Table(title="Vertex AI Search Configuration", show_header=False)
    cfg_table.add_column("Key", style="cyan")
    cfg_table.add_column("Value", style="white")
    cfg_table.add_row("Data store", config.vertex_datastore_id)
    cfg_table.add_row("Region", config.google_cloud_location)
    cfg_table.add_row("Model", config.default_worker_model)
    cfg_table.add_row("Auth", "Application Default Credentials (ADC)")
    console.print(cfg_table)

                                   Vertex AI Search Configuration                                   
┌────────────┬─────────────────────────────────────────────────────────────────────────────────────┐
│ Data store │ projects/agentic-ai-evaluation-bootcamp/locations/global/collections/default_colle… │
│ Region     │ us-central1                                                                         │
│ Model      │ gemini-2.5-flash                                                                    │
│ Auth       │ Application Default Credentials (ADC)                                               │
└────────────┴─────────────────────────────────────────────────────────────────────────────────────┘

## 2. Querying the Knowledge Base

`vertex_search(query)` sends a query to the data store and returns a grounded answer.
The model only has access to the documents in your data store — it cannot draw on the
public web or its training data to answer.

The result dict always contains:

| Key | Type | Description |
|-----|------|-------------|
| `status` | `str` | `"success"` or `"error"` |
| `summary` | `str` | Grounded answer drawn from retrieved documents |
| `sources` | `list[dict]` | Retrieved documents — each with `title` and `uri` |
| `source_count` | `int` | Number of documents cited |
| `error` | `str` | Error message (only present when `status == "error"`) |

In [3]:
result = await vertex_search("What are the pricing tiers for Northstar Analytics?")

console.print(result)

2026-03-02 10:03:35,229 INFO google_genai._api_client: The user provided project/location will take precedence over the Vertex AI API key from the environment variable.
2026-03-02 10:03:35,254 INFO google_genai.models: AFC is enabled with max remote calls: 10.


{
    'status': 'success',
    'summary': 'Northstar Analytics offers three distinct pricing tiers:\n\n*   **Starter:** Priced 
at $299 per month, this tier supports up to 5 users. It includes 500 GB of data storage and allows 
for 200 API requests per minute.\n*   **Professional:** This tier costs $899 per month and 
accommodates up to 25 users. It provides 2 TB of data storage and an API rate limit of 1,200 
requests per minute.\n*   **Enterprise:** At $2,450 per month, the Enterprise tier offers unlimited 
users. It comes with 10 TB of data storage, an API rate limit of 8,000 requests per minute, and a 
99.95% uptime guarantee with a maximum incident response time of 15 minutes.\n\nAll data storage for
Northstar Analytics is hosted exclusively in Canadian data centers.',
    'sources': [
        {
            'title': 'Pricing Tiers',
            'uri': 
'projects/338431745290/locations/global/collections/default_collection/dataStores/vertex-search-test
-v2/branches/0/documents/northstar-pricing'
        },
        {
            'title': 'API Rate Limits',
            'uri': 
'projects/338431745290/locations/global/collections/default_collection/dataStores/vertex-search-test
-v2/branches/0/documents/northstar-api-limit'
        },
        {
            'title': 'Storage Limits',
            'uri': 
'projects/338431745290/locations/global/collections/default_collection/dataStores/vertex-search-test
-v2/branches/0/documents/northstar-storage'
        },
        {
            'title': 'Service Level Agreement',
            'uri': 
'projects/338431745290/locations/global/collections/default_collection/dataStores/vertex-search-test
-v2/branches/0/documents/northstar-sla'
        },
        {
            'title': 'Company Founding',
            'uri': 
'projects/338431745290/locations/global/collections/default_collection/dataStores/vertex-search-test
-v2/branches/0/documents/northstar-founding'
        }
    ],
    'source_count': 5
}

In [4]:
# Display the result in a more readable format
console.print(
    Panel(
        result["summary"],
        title="Answer",
        border_style="cyan",
    )
)

if result["sources"]:
    src_table = Table(title=f"Sources ({result['source_count']} retrieved)")
    src_table.add_column("#", style="dim", width=3)
    src_table.add_column("Title", style="cyan")
    src_table.add_column("Document", style="dim")

    for i, src in enumerate(result["sources"], 1):
        # URI is the full document resource name — the last segment is the document ID
        doc_id = src["uri"].split("/")[-1] if src["uri"] else ""
        src_table.add_row(str(i), src["title"], doc_id)

    console.print(src_table)

╭───────────────────────────────────────────── Answer ─────────────────────────────────────────────╮
│ Northstar Analytics offers three distinct pricing tiers:                                         │
│                                                                                                  │
│ *   **Starter:** Priced at $299 per month, this tier supports up to 5 users. It includes 500 GB  │
│ of data storage and allows for 200 API requests per minute.                                      │
│ *   **Professional:** This tier costs $899 per month and accommodates up to 25 users. It         │
│ provides 2 TB of data storage and an API rate limit of 1,200 requests per minute.                │
│ *   **Enterprise:** At $2,450 per month, the Enterprise tier offers unlimited users. It comes    │
│ with 10 TB of data storage, an API rate limit of 8,000 requests per minute, and a 99.95% uptime  │
│ guarantee with a maximum incident response time of 15 minutes.                                   │
│                                                                                                  │
│ All data storage for Northstar Analytics is hosted exclusively in Canadian data centers.         │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

                 Sources (5 retrieved)                 
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Title                   ┃ Document            ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Pricing Tiers           │ northstar-pricing   │
│ 2   │ API Rate Limits         │ northstar-api-limit │
│ 3   │ Storage Limits          │ northstar-storage   │
│ 4   │ Service Level Agreement │ northstar-sla       │
│ 5   │ Company Founding        │ northstar-founding  │
└─────┴─────────────────────────┴─────────────────────┘

## 3. Grounding in Practice

The key property of this tool: the model's answer is **constrained to what's in the data store**.
Below we run several queries and verify that specific values from the documents appear in the answers.

The test data store contains five documents about a fictional company, **Northstar Analytics**,
with invented values (prices, SLA figures, rate limits) that don't exist in the model's training data.
If those values appear in the answer, they came from the data store.

In [5]:
queries = [
    ("Professional tier monthly price", "What is the monthly price for the Professional tier?"),
    ("Enterprise SLA uptime", "What uptime does the Enterprise plan guarantee?"),
    ("Enterprise API rate limit", "How many API requests per minute does the Enterprise tier allow?"),
    ("Storage on Starter tier", "How much storage does the Starter tier include?"),
    ("Company founders", "Who founded Northstar Analytics and when?"),
]

results_table = Table(title="Grounded Query Results")
results_table.add_column("Query", style="cyan", width=28)
results_table.add_column("Summary", style="white", width=55)
results_table.add_column("Sources", style="dim", justify="right", width=7)

for label, query in queries:
    r = await vertex_search(query)
    summary = r["summary"][:110] + "..." if len(r["summary"]) > 110 else r["summary"]
    status = r["source_count"]
    results_table.add_row(label, summary, str(status))

console.print(results_table)

2026-03-02 10:03:39,442 INFO google_genai._api_client: The user provided project/location will take precedence over the Vertex AI API key from the environment variable.
2026-03-02 10:03:39,456 INFO google_genai.models: AFC is enabled with max remote calls: 10.
2026-03-02 10:03:42,837 INFO google_genai._api_client: The user provided project/location will take precedence over the Vertex AI API key from the environment variable.
2026-03-02 10:03:42,862 INFO google_genai.models: AFC is enabled with max remote calls: 10.
2026-03-02 10:03:45,135 INFO google_genai._api_client: The user provided project/location will take precedence over the Vertex AI API key from the environment variable.
2026-03-02 10:03:45,166 INFO google_genai.models: AFC is enabled with max remote calls: 10.
2026-03-02 10:03:47,721 INFO google_genai._api_client: The user provided project/location will take precedence over the Vertex AI API key from the environment variable.
2026-03-02 10:03:47,746 INFO google_genai.models

                                       Grounded Query Results                                       
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Query                        ┃ Summary                                                 ┃ Sources ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ Professional tier monthly    │ The monthly price for the Professional tier from        │       3 │
│ price                        │ Northstar Analytics is $899. This tier allows for up to │         │
│                              │ 25 us...                                                │         │
│ Enterprise SLA uptime        │ The Enterprise plan guarantees 99.95% uptime.           │       1 │
│ Enterprise API rate limit    │ The Enterprise tier allows 8,000 API requests per       │       3 │
│                              │ minute.                                                 │         │
│ Storage on Starter tier      │ The Starter tier for Northstar Analytics includes 500   │       3 │
│                              │ GB of data storage.                                     │         │
│ Company founders             │ Northstar Analytics was founded in 2019 by Dr. Elena    │       4 │
│                              │ Vasquez and Marcus Thibodeau.                           │         │
└──────────────────────────────┴─────────────────────────────────────────────────────────┴─────────┘

### What happens for out-of-scope questions?

If the question cannot be answered from the data store, the model says so rather than hallucinating.

In [6]:
oos_result = await vertex_search("What is the capital of France?")

console.print(
    Panel(
        f"[bold]Summary:[/bold] {oos_result['summary']}\n[bold]Sources:[/bold] {oos_result['source_count']}",
        title="Out-of-scope question",
        border_style="yellow",
    )
)

2026-03-02 10:03:53,751 INFO google_genai._api_client: The user provided project/location will take precedence over the Vertex AI API key from the environment variable.
2026-03-02 10:03:53,771 INFO google_genai.models: AFC is enabled with max remote calls: 10.


╭───────────────────────────────────── Out-of-scope question ──────────────────────────────────────╮
│ Summary:                                                                                         │
│ Sources: 0                                                                                       │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

## 4. Using the Tool Inside an Agent

`create_vertex_search_tool()` wraps `vertex_search` as an ADK `FunctionTool` that can be
passed directly to a `KnowledgeGroundedAgent` or any other ADK agent.

The tool's docstring becomes the model's description of what the tool does — write it clearly.

In [7]:
tool = create_vertex_search_tool()

console.print(f"Type:          [cyan]{type(tool).__name__}[/cyan]")
console.print(f"Function name: [cyan]{tool.func.__name__}[/cyan]")
console.print()
console.print(Panel(tool.func.__doc__ or "", title="Tool description (seen by the model)", border_style="dim"))

Type:          FunctionTool

Function name: vertex_search

╭────────────────────────────── Tool description (seen by the model) ──────────────────────────────╮
│ Search the custom knowledge base and return grounded results with citations.                     │
│                                                                                                  │
│         Use this tool to find information from internal documents and knowledge bases.           │
│         Results are grounded directly from retrieved document content — the summary              │
│         is more reliable than web search snippets and no separate fetch step is needed.          │
│                                                                                                  │
│         Parameters                                                                               │
│         ----------                                                                               │
│         query : str                                                                              │
│             The search query. Be specific and include key terms.                                 │
│                                                                                                  │
│         Returns                                                                                  │
│         -------                                                                                  │
│         dict                                                                                     │
│             Search results with the following keys:                                              │
│                                                                                                  │
│             - **status** (str): ``"success"`` or ``"error"``                                     │
│             - **summary** (str): Grounded answer from the knowledge base                         │
│             - **sources** (list): Each with ``'title'`` and ``'uri'``                            │
│             - **source_count** (int): Number of source documents retrieved                       │
│             - **error** (str): Error message (error case only)                                   │
│                                                                                                  │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

### Passing the tool to an agent

Replace `create_google_search_tool()` with `create_vertex_search_tool()` in your agent
setup, or add it alongside the web search tool:

```python
from aieng.agent_evals.tools import create_vertex_search_tool
from google.adk.agents import Agent

agent = Agent(
    model="gemini-2.5-flash",
    tools=[
        create_vertex_search_tool(),  # custom knowledge base
    ],
    instruction="Use vertex_search to answer questions from the knowledge base.",
)
```

The agent will call `vertex_search` when it determines the question can be answered
from the knowledge base, and the grounded answer is returned directly — no separate
fetch step required.

## Summary

| | `google_search` + `web_fetch` | `vertex_search` |
|---|---|---|
| **Data source** | Public web | Your private data store |
| **Auth** | API key | ADC (automatic on GCE) |
| **Steps to get an answer** | Search → Fetch → Read | Single call |
| **Grounding** | Model reads fetched HTML | Model reads retrieved document chunks |
| **Out-of-scope queries** | Searches the web anyway | Tells the user it doesn't know |

Use `vertex_search` when you want the agent to be strictly grounded in a controlled
document set. Use `google_search` when the agent needs to find current information
from the public internet.